In [1]:
# ============================================================================
# CUSTOMER CHURN PREDICTION SYSTEM - COMPLETE CODE
# ============================================================================

# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_auc_score, roc_curve)
from sklearn.utils import class_weight
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("CUSTOMER CHURN PREDICTION SYSTEM")
print("="*70)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n" + "="*70)
print("STEP 1: LOADING DATA")
print("="*70)

# Load the dataset (UPDATE THIS PATH TO YOUR FILE LOCATION)
df = pd.read_csv("C:/Users/mohansai/OneDrive/Desktop/PROJECTS/Customer Churn/archive/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())

# ============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================================
print("\n" + "="*70)
print("STEP 2: EXPLORATORY DATA ANALYSIS")
print("="*70)

# Basic info
print(f"\nDataset Info:")
print(df.info())

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

# Churn distribution
print(f"\nChurn distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {(df['Churn'].value_counts()['Yes'] / len(df)) * 100:.2f}%")

# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'coral'])
axes[0].set_title('Churn Distribution')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No', 'Yes'], rotation=0)

df['Churn'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                 colors=['skyblue', 'coral'])
axes[1].set_title('Churn Percentage')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 3: DATA PREPROCESSING
# ============================================================================
print("\n" + "="*70)
print("STEP 3: DATA PREPROCESSING")
print("="*70)

# Create a clean copy
df_clean = df.copy()

# Drop customerID (not useful for prediction)
df_clean = df_clean.drop(['customerID'], axis=1)
print("\n✓ Dropped customerID column")

# Handle TotalCharges (convert to numeric)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
missing_total = df_clean['TotalCharges'].isnull().sum()
print(f"\n✓ Converted TotalCharges to numeric ({missing_total} missing values)")

# Fill missing TotalCharges with median
df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median(), inplace=True)
print("✓ Filled missing TotalCharges with median")

# Encode binary variables
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0})
print(f"\n✓ Encoded {len(binary_cols)} binary columns")

# Handle service columns with 'No phone service' or 'No internet service'
service_map = {'No': 0, 'Yes': 1, 'No phone service': 0, 'No internet service': 0}
service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in service_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(service_map)
print(f"✓ Encoded {len(service_cols)} service columns")

# One-hot encode remaining categorical columns
cat_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
print(f"✓ One-hot encoded {len(cat_cols)} categorical columns")

# Convert Churn to binary (do this LAST to avoid data leakage)
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})
print("✓ Converted Churn to binary")

print(f"\nFinal preprocessed shape: {df_clean.shape}")
print(f"Final columns: {len(df_clean.columns)}")

# ============================================================================
# STEP 4: PREPARE FEATURES AND TARGET
# ============================================================================
print("\n" + "="*70)
print("STEP 4: PREPARING FEATURES AND TARGET")
print("="*70)

# Separate features (X) and target (y)
X = df_clean.drop('Churn', axis=1)
y = df_clean['Churn']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Churn rate in train: {(y_train.sum() / len(y_train)) * 100:.2f}%")
print(f"Churn rate in test: {(y_test.sum() / len(y_test)) * 100:.2f}%")

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\n✓ Features scaled using StandardScaler")

# ============================================================================
# STEP 5: TRAIN BASELINE MODELS
# ============================================================================
print("\n" + "="*70)
print("STEP 5: TRAINING BASELINE MODELS")
print("="*70)

results = {}

# Model 1: Logistic Regression
print("\n[1/4] Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_test, lr_pred),
    'roc_auc': roc_auc_score(y_test, lr_proba),
    'model': lr_model,
    'predictions': lr_pred,
    'probabilities': lr_proba,
    'confusion_matrix': confusion_matrix(y_test, lr_pred)
}
print(f"✓ Accuracy: {results['Logistic Regression']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {results['Logistic Regression']['roc_auc']:.4f}")

# Model 2: Random Forest
print("\n[2/4] Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, rf_pred),
    'roc_auc': roc_auc_score(y_test, rf_proba),
    'model': rf_model,
    'predictions': rf_pred,
    'probabilities': rf_proba,
    'confusion_matrix': confusion_matrix(y_test, rf_pred)
}
print(f"✓ Accuracy: {results['Random Forest']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {results['Random Forest']['roc_auc']:.4f}")

# Model 3: Gradient Boosting
print("\n[3/4] Training Gradient Boosting...")
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_proba = gb_model.predict_proba(X_test)[:, 1]

results['Gradient Boosting'] = {
    'accuracy': accuracy_score(y_test, gb_pred),
    'roc_auc': roc_auc_score(y_test, gb_proba),
    'model': gb_model,
    'predictions': gb_pred,
    'probabilities': gb_proba,
    'confusion_matrix': confusion_matrix(y_test, gb_pred)
}
print(f"✓ Accuracy: {results['Gradient Boosting']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {results['Gradient Boosting']['roc_auc']:.4f}")

# Model 4: Decision Tree
print("\n[4/4] Training Decision Tree...")
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

results['Decision Tree'] = {
    'accuracy': accuracy_score(y_test, dt_pred),
    'roc_auc': roc_auc_score(y_test, dt_proba),
    'model': dt_model,
    'predictions': dt_pred,
    'probabilities': dt_proba,
    'confusion_matrix': confusion_matrix(y_test, dt_pred)
}
print(f"✓ Accuracy: {results['Decision Tree']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {results['Decision Tree']['roc_auc']:.4f}")

# ============================================================================
# STEP 6: MODEL COMPARISON
# ============================================================================
print("\n" + "="*70)
print("STEP 6: BASELINE MODEL COMPARISON")
print("="*70)

comparison_data = {
    model_name: {
        'accuracy': data['accuracy'],
        'roc_auc': data['roc_auc']
    }
    for model_name, data in results.items()
}

comparison_df = pd.DataFrame(comparison_data).T.sort_values('roc_auc', ascending=False)
print("\n")
print(comparison_df)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparison_df['accuracy'].plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].grid(axis='x', alpha=0.3)

comparison_df['roc_auc'].plot(kind='barh', ax=axes[1], color='lightcoral')
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_title('Model ROC-AUC Comparison')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 7: HANDLE CLASS IMBALANCE
# ============================================================================
print("\n" + "="*70)
print("STEP 7: HANDLING CLASS IMBALANCE")
print("="*70)

# Calculate class weights
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"\nClass weights: {class_weight_dict}")
print(f"Churn rate: {(y_train.sum() / len(y_train)) * 100:.2f}%")

balanced_results = {}

# Balanced Model 1: Logistic Regression
print("\n[1/3] Training Balanced Logistic Regression...")
lr_balanced = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42, class_weight='balanced')
lr_balanced.fit(X_train_scaled, y_train)
lr_bal_pred = lr_balanced.predict(X_test_scaled)
lr_bal_proba = lr_balanced.predict_proba(X_test_scaled)[:, 1]

balanced_results['LR Balanced'] = {
    'accuracy': accuracy_score(y_test, lr_bal_pred),
    'roc_auc': roc_auc_score(y_test, lr_bal_proba),
    'confusion_matrix': confusion_matrix(y_test, lr_bal_pred),
    'model': lr_balanced
}
print(f"✓ Accuracy: {balanced_results['LR Balanced']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {balanced_results['LR Balanced']['roc_auc']:.4f}")

# Balanced Model 2: Random Forest
print("\n[2/3] Training Balanced Random Forest...")
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, 
                                     max_depth=10, class_weight='balanced')
rf_balanced.fit(X_train, y_train)
rf_bal_pred = rf_balanced.predict(X_test)
rf_bal_proba = rf_balanced.predict_proba(X_test)[:, 1]

balanced_results['RF Balanced'] = {
    'accuracy': accuracy_score(y_test, rf_bal_pred),
    'roc_auc': roc_auc_score(y_test, rf_bal_proba),
    'confusion_matrix': confusion_matrix(y_test, rf_bal_pred),
    'model': rf_balanced
}
print(f"✓ Accuracy: {balanced_results['RF Balanced']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {balanced_results['RF Balanced']['roc_auc']:.4f}")

# Balanced Model 3: Gradient Boosting
print("\n[3/3] Training Balanced Gradient Boosting...")
gb_balanced = GradientBoostingClassifier(n_estimators=100, random_state=42, 
                                         max_depth=5, learning_rate=0.1)
sample_weights = np.ones(len(y_train))
sample_weights[y_train == 1] = class_weights[1] / class_weights[0]
gb_balanced.fit(X_train, y_train, sample_weight=sample_weights)
gb_bal_pred = gb_balanced.predict(X_test)
gb_bal_proba = gb_balanced.predict_proba(X_test)[:, 1]

balanced_results['GB Balanced'] = {
    'accuracy': accuracy_score(y_test, gb_bal_pred),
    'roc_auc': roc_auc_score(y_test, gb_bal_proba),
    'confusion_matrix': confusion_matrix(y_test, gb_bal_pred),
    'model': gb_balanced
}
print(f"✓ Accuracy: {balanced_results['GB Balanced']['accuracy']:.4f}")
print(f"✓ ROC-AUC: {balanced_results['GB Balanced']['roc_auc']:.4f}")

# ============================================================================
# STEP 8: COMPARE ORIGINAL VS BALANCED MODELS
# ============================================================================
print("\n" + "="*70)
print("STEP 8: ORIGINAL VS BALANCED MODEL COMPARISON")
print("="*70)

# Combine results
all_comparison = {}
for name, data in results.items():
    all_comparison[f"{name} (Original)"] = {
        'accuracy': data['accuracy'],
        'roc_auc': data['roc_auc']
    }
for name, data in balanced_results.items():
    all_comparison[name] = {
        'accuracy': data['accuracy'],
        'roc_auc': data['roc_auc']
    }

final_comparison_df = pd.DataFrame(all_comparison).T.sort_values('roc_auc', ascending=False)
print("\n")
print(final_comparison_df)

# Recall comparison for churners
print("\n" + "="*70)
print("RECALL COMPARISON FOR CHURNERS (Most Important Metric)")
print("="*70)

recall_comparison = {}

for name, data in results.items():
    cm = data['confusion_matrix']
    recall = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    recall_comparison[f"{name} (Original)"] = recall

for name, data in balanced_results.items():
    cm = data['confusion_matrix']
    recall = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    recall_comparison[name] = recall

recall_df = pd.DataFrame(recall_comparison.items(), columns=['Model', 'Recall for Churners'])
recall_df = recall_df.sort_values('Recall for Churners', ascending=False)
print("\n")
print(recall_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

final_comparison_df['roc_auc'].plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_xlabel('ROC-AUC Score')
axes[0].set_title('Model ROC-AUC Comparison (All Models)')
axes[0].grid(axis='x', alpha=0.3)

recall_df_plot = recall_df.set_index('Model')
recall_df_plot.plot(kind='barh', ax=axes[1], color='lightcoral', legend=False)
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_title('Recall for Churners - Business Priority!')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 9: FEATURE IMPORTANCE ANALYSIS
# ============================================================================
print("\n" + "="*70)
print("STEP 9: FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Use Random Forest for feature importance
best_rf_model = balanced_results['RF Balanced']['model']

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Visualize top 15
plt.figure(figsize=(10, 8))
top_15 = feature_importance.head(15)
plt.barh(range(len(top_15)), top_15['importance'], color='steelblue')
plt.yticks(range(len(top_15)), top_15['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Features Driving Customer Churn')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Cumulative importance
feature_importance['cumulative'] = feature_importance['importance'].cumsum()
print(f"\n📊 Top 5 features:  {feature_importance.head(5)['cumulative'].iloc[-1]:.1%}")
print(f"📊 Top 10 features: {feature_importance.head(10)['cumulative'].iloc[-1]:.1%}")
print(f"📊 Top 20 features: {feature_importance.head(20)['cumulative'].iloc[-1]:.1%}")

# ============================================================================
# STEP 10: FINAL MODEL SELECTION & EVALUATION
# ============================================================================
print("\n" + "="*70)
print("STEP 10: FINAL MODEL SELECTION")
print("="*70)

# Select best model based on ROC-AUC
best_model_name = recall_df.iloc[0]['Model']
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Recall for Churners: {recall_df.iloc[0]['Recall for Churners']:.2%}")

# Get the best model
if 'LR' in best_model_name:
    best_model = balanced_results['LR Balanced']['model']
    best_pred = lr_bal_pred
    best_proba = lr_bal_proba
elif 'RF' in best_model_name:
    best_model = balanced_results['RF Balanced']['model']
    best_pred = rf_bal_pred
    best_proba = rf_bal_proba
else:
    best_model = balanced_results['GB Balanced']['model']
    best_pred = gb_bal_pred
    best_proba = gb_bal_proba

# Final evaluation
print("\n" + "="*70)
print("FINAL MODEL EVALUATION")
print("="*70)
print(f"\nAccuracy: {accuracy_score(y_test, best_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, best_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_pred))
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['No Churn', 'Churn']))

# Final visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm_final = confusion_matrix(y_test, best_pred)
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix - {best_model_name}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, best_proba)
axes[1].plot(fpr, tpr, label=f'ROC (AUC = {roc_auc_score(y_test, best_proba):.3f})', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=2)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve - {best_model_name}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)
print(f"""
✓ Dataset: {df.shape[0]} customers, {df.shape[1]} original features
✓ Churn Rate: {(y.sum() / len(y)) * 100:.2f}%
✓ Models Trained: 7 (4 original + 3 balanced)
✓ Best Model: {best_model_name}
✓ Best Recall for Churners: {recall_df.iloc[0]['Recall for Churners']:.2%}
✓ ROC-AUC: {roc_auc_score(y_test, best_proba):.4f}

Key Findings:
- Balanced models significantly improve churn detection
- Top features: {', '.join(feature_importance.head(3)['feature'].tolist())}
- Ready for deployment and business use

Next Steps:
1. Deploy model to production
2. Create monitoring dashboard
3. Implement retention campaigns for high-risk customers
4. Measure business impact (reduced churn rate)
""")

print("="*70)
print("CHURN PREDICTION SYSTEM - COMPLETE!")
print("="*70)

CUSTOMER CHURN PREDICTION SYSTEM

STEP 1: LOADING DATA

Dataset loaded successfully!
Shape: (7043, 21)

First few rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No

<Figure size 1200x400 with 2 Axes>


STEP 3: DATA PREPROCESSING

✓ Dropped customerID column

✓ Converted TotalCharges to numeric (11 missing values)
✓ Filled missing TotalCharges with median

✓ Encoded 5 binary columns
✓ Encoded 7 service columns
✓ One-hot encoded 3 categorical columns
✓ Converted Churn to binary

Final preprocessed shape: (7043, 24)
Final columns: 24

STEP 4: PREPARING FEATURES AND TARGET

Features shape: (7043, 23)
Target shape: (7043,)

Target distribution:
0    5174
1    1869
Name: Churn, dtype: int64

Train set size: (5634, 23)
Test set size: (1409, 23)
Churn rate in train: 26.54%
Churn rate in test: 26.54%

✓ Features scaled using StandardScaler

STEP 5: TRAINING BASELINE MODELS

[1/4] Training Logistic Regression...
✓ Accuracy: 0.8070
✓ ROC-AUC: 0.8417

[2/4] Training Random Forest...
✓ Accuracy: 0.7999
✓ ROC-AUC: 0.8417

[3/4] Training Gradient Boosting...
✓ Accuracy: 0.7984
✓ ROC-AUC: 0.8355

[4/4] Training Decision Tree...
✓ Accuracy: 0.7942
✓ ROC-AUC: 0.8284

STEP 6: BASELINE MODEL COMPARISON

<Figure size 1400x500 with 2 Axes>


STEP 7: HANDLING CLASS IMBALANCE

Class weights: {0: 0.6805991785455424, 1: 1.8842809364548494}
Churn rate: 26.54%

[1/3] Training Balanced Logistic Regression...
✓ Accuracy: 0.7381
✓ ROC-AUC: 0.8413

[2/3] Training Balanced Random Forest...
✓ Accuracy: 0.7658
✓ ROC-AUC: 0.8411

[3/3] Training Balanced Gradient Boosting...
✓ Accuracy: 0.7530
✓ ROC-AUC: 0.8346

STEP 8: ORIGINAL VS BALANCED MODEL COMPARISON


                                accuracy   roc_auc
Logistic Regression (Original)  0.806955  0.841740
Random Forest (Original)        0.799858  0.841669
LR Balanced                     0.738112  0.841269
RF Balanced                     0.765791  0.841099
Gradient Boosting (Original)    0.798439  0.835546
GB Balanced                     0.753016  0.834575
Decision Tree (Original)        0.794180  0.828358

RECALL COMPARISON FOR CHURNERS (Most Important Metric)


                          Model  Recall for Churners
                    LR Balanced             0.780749
                

...


STEP 9: FEATURE IMPORTANCE ANALYSIS

Top 20 Most Important Features:
                               feature  importance
                                tenure    0.179510
                          TotalCharges    0.135324
                        MonthlyCharges    0.127863
                     Contract_Two year    0.106300
           InternetService_Fiber optic    0.077582
        PaymentMethod_Electronic check    0.052479
                     Contract_One year    0.043045
                    InternetService_No    0.038206
                        OnlineSecurity    0.029397
                      PaperlessBilling    0.028982
                           TechSupport    0.023094
                            Dependents    0.017081
                          OnlineBackup    0.015881
                                gender    0.015346
                               Partner    0.015300
                         MultipleLines    0.014140
                         SeniorCitizen    0.013270
            

<Figure size 1000x800 with 1 Axes>


📊 Top 5 features:  62.7%
📊 Top 10 features: 81.9%
📊 Top 20 features: 97.2%

STEP 10: FINAL MODEL SELECTION

🏆 BEST MODEL: LR Balanced
   Recall for Churners: 78.07%

FINAL MODEL EVALUATION

Accuracy: 0.7381
ROC-AUC Score: 0.8413

Confusion Matrix:
[[748 287]
 [ 82 292]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.90      0.72      0.80      1035
       Churn       0.50      0.78      0.61       374

   micro avg       0.74      0.74      0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



<Figure size 1400x500 with 3 Axes>


PROJECT SUMMARY

✓ Dataset: 7043 customers, 21 original features
✓ Churn Rate: 26.54%
✓ Models Trained: 7 (4 original + 3 balanced)
✓ Best Model: LR Balanced
✓ Best Recall for Churners: 78.07%
✓ ROC-AUC: 0.8413

Key Findings:
- Balanced models significantly improve churn detection
- Top features: tenure, TotalCharges, MonthlyCharges
- Ready for deployment and business use

Next Steps:
1. Deploy model to production
2. Create monitoring dashboard
3. Implement retention campaigns for high-risk customers
4. Measure business impact (reduced churn rate)

CHURN PREDICTION SYSTEM - COMPLETE!


In [8]:
import pickle
# ============================================================================
# SAVE MODEL FOR DEPLOYMENT
# ============================================================================
print("\n" + "="*70)
print("SAVING MODEL FOR DEPLOYMENT")
print("="*70)

# Import the system class (make sure churn_model.py is in the same directory)
from churn_model import ChurnPredictionSystem

# Create the prediction system
churn_system = ChurnPredictionSystem(
    model=best_model,  # Your best model from training
    scaler=scaler,     # Your fitted scaler
    feature_names=X.columns.tolist()  # List of feature names
)

# Save to pickle file
with open("churn_system.pkl", "wb") as f:
    pickle.dump(churn_system, f)

print("✅ Model saved as 'churn_system.pkl'")
print(f"✅ Features saved: {len(X.columns)}")
print("\nNow you can run: streamlit run app.py")


SAVING MODEL FOR DEPLOYMENT
✅ Model saved as 'churn_system.pkl'
✅ Features saved: 23

Now you can run: streamlit run app.py
